**Retrievers Intro**

RETRIEVERS: bht imp h ragg h bhaiii

what? component in langchain that fetches relevant doc from data source in response to query
      data source kch bhi hsokta h api, vector store aisa kch now query aai....retriever data source me relevant data miljata h and then fetch krke dedeta h

ek retriever ni hota h bht h for multiple use cases...sb runnables h jse prompt....toh chaining me plugin krr skte ho

TYPES:
2 categories: on data storeee....

wikipedia store: to ye wikipedia p store krta h
vector store based retriever
one way of differentiation ki data sourcee kya h

another serach strategy way: mmr, multiquery, contextual compression etc...alg searching tarika perform

**Wikipedia Retriever**


queries wikipedia api to fetch relevant content..suppose ipl to uski api fetch krdeaa
query di-> sends query to wiki api-> retrieves relevant articles -> returns them as langchain doc objects

In [ ]:
# Install libraries
!pip install -q langchain langchain-community wikipedia

# Import
from langchain_community.retrievers import WikipediaRetriever

# Create retriever
retriever = WikipediaRetriever(
    top_k_results=2,     # number of articles to retrieve
    lang="en"
)

# Search Wikipedia
docs = retriever.invoke("Artificial Intelligence")

# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n{'='*20} RESULT {i+1} {'='*20}")
    print("Title:", doc.metadata["title"])
    print("\nContent:")
    print(doc.page_content[:1000])  # first 1000 characters

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_3011/2493739173.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever



==================== RESULT 1 ====================
Title: Artificial intelligence

Content:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural langua

**Vector Search Retriever**

search and fetch docs from vector store based on semantic similarity using vector embeddings
store doc in vector store -> each into vector using embedding model -> query also turned into vector, compares query vector with others and retrieves top k

document bhi bnaan h, embedding bnaii....bnaya and add krdiaa.....vectorstore.as_retriever k=2
but yee kaam to vector store bhi dera thaa? but .similarity_search etcc to retriever bnayaa hi kyuu?....ans ye h yes capability h ki ek hi strategy se compare krega but what if different strategy...but retrievers h toh dif diff search strategy

ye code to vanilla...aage advanced search strategy bhi krskte hoo.....

In [ ]:
# Install libraries
!pip install -qU langchain chromadb langchain-google-genai

# Imports
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from google.colab import userdata

# Gemini API key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# Embedding model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

# ---------------- DOCUMENTS ----------------

docs = [
    Document(
        page_content="Artificial Intelligence helps machines think and solve problems."
    ),

    Document(
        page_content="Machine Learning is a subset of AI where systems learn from data."
    ),

    Document(
        page_content="Cricket is one of the most popular sports in India."
    ),

    Document(
        page_content="Football is played worldwide and famous players include Messi and Ronaldo."
    ),

    Document(
        page_content="Java is widely used in backend development and DSA."
    )
]

# ---------------- CREATE VECTOR DB ----------------

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="sample_db"
)

# ---------------- CREATE RETRIEVER ----------------

retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}   # top 2 results
)

# ---------------- SEARCH ----------------

query = "What is used for learning from data?"

results = retriever.invoke(query)

# Print retrieved results
print(f"\nQuery: {query}")

for i, doc in enumerate(results):
    print(f"\n{'='*20} RESULT {i+1} {'='*20}")
    print(doc.page_content)

/tmp/ipykernel_5212/1842614360.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma



Query: What is used for learning from data?

==================== RESULT 1 ====================
Machine Learning is a subset of AI where systems learn from data.

==================== RESULT 2 ====================
Artificial Intelligence helps machines think and solve problems.


**MMR SEARCH STRATEGY**

problem h : maanlo 5 docu h 2 same bt alg alg lines me likhdi h....normal me top 3 dikkt ki 2 same bhjdegaaa
mmr yahi krta h not only doc relevant ho but also ek dusre ke algg + non redundant...isse different perspective milta h, sbse zyada relevant, 2nd not only relevant but also dissimilar

document bnayaa, vector store bnata, vector store as retriever, search type and search_kwargs and lambda 1 kroge to normal jse behave and 0 rkhoge to bht zyadaa diverseee so balance me rkhoww


In [ ]:
import shutil
import os

# Delete old DB
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

docs = [

    # Similar healthcare docs
    Document(page_content="Artificial Intelligence is used in healthcare for disease prediction."),
    Document(page_content="AI helps doctors predict diseases using patient data."),
    Document(page_content="Healthcare systems use Artificial Intelligence for diagnosis."),
    Document(page_content="AI improves disease detection in hospitals."),
    Document(page_content="Machine learning predicts illnesses in hospitals."),
    Document(page_content="AI helps doctors identify diseases early."),

    # Different AI applications
    Document(page_content="Artificial Intelligence powers self-driving cars."),
    Document(page_content="AI helps banks detect fraud."),
    Document(page_content="AI powers Netflix recommendation systems."),
    Document(page_content="Natural Language Processing powers chatbots."),
    Document(page_content="AI improves cybersecurity systems."),
]

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="fresh_mmr_demo",
    persist_directory="./chroma_db"
)

query = "Applications of Artificial Intelligence in healthcare"

# NORMAL SEARCH
print("="*50)
print("NORMAL SEARCH")
print("="*50)

results = vector_store.similarity_search(query, k=5)

for i, doc in enumerate(results):
    print(f"\n{i+1}. {doc.page_content}")

# MMR SEARCH
print("\n" + "="*50)
print("MMR SEARCH lambda=0.2")
print("="*50)

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 10,
        "lambda_mult": 0.2
    }
)

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n{i+1}. {doc.page_content}")

NORMAL SEARCH

1. Artificial Intelligence is used in healthcare for disease prediction.

2. Healthcare systems use Artificial Intelligence for diagnosis.

3. AI helps doctors predict diseases using patient data.

4. AI helps doctors identify diseases early.

5. AI improves disease detection in hospitals.

MMR SEARCH lambda=0.2

1. Artificial Intelligence is used in healthcare for disease prediction.

2. AI improves cybersecurity systems.

3. Artificial Intelligence powers self-driving cars.

4. AI helps banks detect fraud.

5. AI powers Netflix recommendation systems.


**Multi query retriever**

manlo vector store h koii, koi query h and ambiguous and not meaning clear, so isko solve krtaa h kisi trh se query se ambiguity solve krtaa h
query jaegii llm paas -> diverse but relative queries generate kregaa -> 5 questions but not ambiguous -> now 5 queries will give retrievers answers to all questions particularly-> per retriever 5 documents-> mergeeee and then return top 5
so overall ambiguity solve krdegaaa

konsa model and everythiing, retreiver konsa use krogee


In [ ]:
# Install compatible versions
!pip uninstall -y langchain langchain-community -q

!pip install -q \
langchain==0.3.25 \
langchain-community==0.3.24 \
langchain-google-genai \
chromadb

# Imports
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain.retrievers.multi_query import MultiQueryRetriever
from google.colab import userdata

# Gemini API key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# ---------------- LLM ----------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3
)

# ---------------- EMBEDDINGS ----------------

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

# ---------------- DOCUMENTS ----------------

docs = [
    Document(page_content="Artificial Intelligence helps doctors diagnose diseases."),
    Document(page_content="Machine Learning predicts diseases using medical records."),
    Document(page_content="AI is used in hospitals for disease prediction."),
    Document(page_content="Healthcare systems use AI for medical imaging."),
    Document(page_content="Artificial Intelligence powers fraud detection in banking."),
    Document(page_content="Netflix uses recommendation systems powered by AI."),
    Document(page_content="Self-driving cars use Artificial Intelligence."),
    Document(page_content="Natural Language Processing powers chatbots.")
]

# ---------------- VECTOR STORE ----------------

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="multi_query_demo"
)

# ---------------- BASE RETRIEVER ----------------

base_retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

# ---------------- MULTI QUERY RETRIEVER ----------------

retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

# ---------------- QUERY ----------------

query = "Applications of AI in healthcare"

results = retriever.invoke(query)

print(f"\nQuery: {query}")

for i, doc in enumerate(results):
    print(f"\n===== RESULT {i+1} =====")
    print(doc.page_content)


Query: Applications of AI in healthcare

===== RESULT 1 =====
Artificial Intelligence helps doctors diagnose diseases.

===== RESULT 2 =====
Healthcare systems use AI for medical imaging.

===== RESULT 3 =====
AI is used in hospitals for disease prediction.

===== RESULT 4 =====
Machine Learning predicts diseases using medical records.


**Contextual Compression retreival**

advanced retriver improve quality by comprssing doc after retrieval ki us particular document me bhi sirff important important hii data aara ho
document me ek se zyadaa cheezo k bare meee baat and thenn u need only impo data

query given to retriever and vo 2 doc nikaalke degaa and then thsee sent to llm and prompt jaegaa and then nyaa d1 and nya d2 baaki sari cheeze removed
so ye context ko compress krra hotaa h

llm pipelines accuracy,
jb data contains bht kch cheezeee

In [ ]:
# Install libraries
!pip install -qU \
langchain==0.3.27 \
langchain-community==0.3.27 \
langchain-core==0.3.72 \
langchain-google-genai \
chromadb

# Imports
from google.colab import userdata
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# ---------------- API KEY ----------------

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# ---------------- GEMINI MODEL ----------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

# ---------------- EMBEDDINGS ----------------

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

# ---------------- LONG DOCUMENTS ----------------

docs = [

    Document(page_content="""
Artificial Intelligence is transforming many industries.
In healthcare, AI helps doctors diagnose diseases,
predict illnesses, and analyze medical images.

In banking, AI is used for fraud detection,
credit scoring, and customer support chatbots.

Self-driving cars use Artificial Intelligence
for object detection and navigation.

Recommendation systems in Netflix and YouTube
also rely heavily on AI to personalize content.
"""),

    Document(page_content="""
Machine Learning is a subset of Artificial Intelligence.
Healthcare organizations use machine learning models
to predict diseases and assist doctors in treatment planning.

Sports analytics also use AI to improve player performance.
Cybersecurity systems use AI for detecting suspicious activity.
""")
]

# ---------------- VECTOR STORE ----------------

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="compression_demo"
)

# ---------------- BASE RETRIEVER ----------------

base_retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

# ---------------- COMPRESSOR ----------------

compressor = LLMChainExtractor.from_llm(llm)

# ---------------- CONTEXTUAL COMPRESSION RETRIEVER ----------------

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# ---------------- QUERY ----------------

query = "How is AI used in healthcare?"

results = compression_retriever.invoke(query)

print(f"\nQuery: {query}")

for i, doc in enumerate(results):
    print(f"\n===== COMPRESSED RESULT {i+1} =====")
    print(doc.page_content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.72 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.9 which is incompatible.
langgraph 1.2.1 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.72 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you 